# El bueno

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

def formatear_precio(precio_raw):
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    else:
        return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    else:
        return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def extraer_detalle_producto(url, options):
    resultados = []
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h3.attributeAccordionText"))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")
        color_list = soup.find("ul", id="color-list-mobile")
        color_buttons = color_list.find_all("a", class_="atom-color") if color_list else [None]
        size_list = soup.find("ul", id="size-list-container")
        size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]

        combinaciones_unicas = set()
        if color_buttons == [None] and size_buttons == [None]:
            color = ""
            codigo = ""
            tamano = ""
            precio_normal = ""
            precio_desc = ""
            color_tag = soup.find("h3", class_="attributeAccordionText")
            if color_tag and "Color:" in color_tag.text:
                color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                color = color_span.text.strip() if color_span else ""
            codigo_tag = soup.find("p", class_="product-header-subcontainer-information--code")
            if codigo_tag:
                codigo_span = codigo_tag.find("span")
                codigo = codigo_span.text.strip() if codigo_span else ""
            tamano_tag = None
            for h3 in soup.find_all("h3", class_="attributeAccordionText"):
                if "Tamaño:" in h3.text:
                    tamano_tag = h3
                    break
            if tamano_tag:
                tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                tamano = tamano_span.text.strip() if tamano_span else ""
            precio_normal_tag = soup.find("p", class_="a-product__paragraphRegularPrice")
            precio_desc_tag = soup.find("p", class_="a-product__paragraphDiscountPrice")
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            clave = (color, tamano, codigo)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((color, codigo, tamano, precio_normal, precio_desc))
        else:
            for color_btn in color_buttons:
                if color_btn:
                    color_skuid = color_btn.get("data-skuid")
                    try:
                        driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//a[@data-skuid='{color_skuid}']"))
                        time.sleep(2)
                    except Exception:
                        pass
                soup_color = BeautifulSoup(driver.page_source, "html.parser")
                size_list = soup_color.find("ul", id="size-list-container")
                size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]
                for size_btn in size_buttons:
                    if size_btn and '-disabled' not in size_btn.get('class', []):
                        size_skuid = size_btn.get("data-skuid")
                        try:
                            driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//button[@data-skuid='{size_skuid}']"))
                            time.sleep(2)
                        except Exception:
                            pass
                    soup_var = BeautifulSoup(driver.page_source, "html.parser")
                    if soup_var.find("p", class_="a-productInfo_selledByLink"):
                        continue
                    color = ""
                    codigo = ""
                    tamano = ""
                    precio_normal = ""
                    precio_desc = ""
                    color_tag = soup_var.find("h3", class_="attributeAccordionText")
                    if color_tag and "Color:" in color_tag.text:
                        color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                        color = color_span.text.strip() if color_span else ""
                    codigo_tag = soup_var.find("p", class_="product-header-subcontainer-information--code")
                    if codigo_tag:
                        codigo_span = codigo_tag.find("span")
                        codigo = codigo_span.text.strip() if codigo_span else ""
                    tamano_tag = None
                    for h3 in soup_var.find_all("h3", class_="attributeAccordionText"):
                        if "Tamaño:" in h3.text:
                            tamano_tag = h3
                            break
                    if tamano_tag:
                        tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                        tamano = tamano_span.text.strip() if tamano_span else ""
                    precio_normal_tag = soup_var.find("p", class_="a-product__paragraphRegularPrice")
                    precio_desc_tag = soup_var.find("p", class_="a-product__paragraphDiscountPrice")
                    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
                    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
                    clave = (color, tamano, codigo)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((color, codigo, tamano, precio_normal, precio_desc))
        driver.quit()
    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    return resultados

def es_smartwatch(nombre):
    nombre = (nombre or "").lower()
    palabras = ["smartwatch"]
    return any(p in nombre for p in palabras)

options = Options()
#options.add_argument("headless")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data = []
visited_urls = set()

driver = webdriver.Chrome(options=options)
driver.get("https://www.liverpool.com.mx/tienda/celulares/N-S1sLjNksKoG%2BC2c1SDPsHEVNIqwseF0ax1%2Fc4tPD6RI%3D")
time.sleep(10)
soup = BeautifulSoup(driver.page_source, "html.parser")
products = soup.find_all("li", class_="m-product__card")
print(f"Productos encontrados: {len(products)}")
for product in products:
    marca = product.find("h3", class_="a-card-brand")
    nombre = product.find("h3", class_="card-title a-card-description")
    nombre_texto = nombre.text.strip() if nombre else ""
    if es_smartwatch(nombre_texto):
        continue
    precio_normal = product.find("p", class_="a-card-price")
    precio_desc = product.find("p", class_="a-card-discount")
    a_tag = product.find("a", href=True)
    url_cel = ""
    if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
        url_cel = "https://www.liverpool.com.mx" + a_tag["href"]
    color_div = product.find("div", class_="a-plp-color")
    colores = []
    if color_div:
        for span in color_div.find_all("span", class_="atom-color"):
            color_nombre = span.get("title", "")
            color_hex = span.get("data-color", "")
            colores.append((color_nombre, color_hex))
    else:
        colores = [("", "")]
    precio_normal_text = precio_normal.text.strip() if precio_normal else ""
    precio_desc_text = precio_desc.text.strip() if precio_desc else ""
    # Si solo hay precio en descuento, guárdalo en precio_normal y "No discount" en descuento
    if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
        for color_nombre, color_hex in colores:
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_nombre,
                "Capacidad": "",
                "CodigoProducto": extraer_codigo_producto(url_cel),
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": formatear_precio(precio_desc_text),
                "Precio_descuento": "No discount"
            })
    elif ('-' in precio_normal_text or '-' in precio_desc_text) and url_cel not in visited_urls:
        variantes = extraer_detalle_producto(url_cel, options)
        visited_urls.add(url_cel)
        combinaciones_unicas = set()
        for color_det, codigo_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
            clave = (color_det, tamano_det, extraer_sku(url_cel))
            if clave in combinaciones_unicas:
                continue
            combinaciones_unicas.add(clave)
            # Si solo hay precio en descuento, guárdalo en precio_normal y "No discount" en descuento
            if not precio_normal_det and precio_desc_det:
                precio_normal_final = formatear_precio(precio_desc_det)
                precio_desc_final = "No discount"
            else:
                precio_normal_final = formatear_precio(precio_normal_det)
                precio_desc_final = formatear_precio(precio_desc_det) if precio_desc_det else "No discount"
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_det,
                "Capacidad": tamano_det,
                "CodigoProducto": codigo_det,
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": precio_normal_final,
                "Precio_descuento": precio_desc_final
            })
    else:
        for color_nombre, color_hex in colores:
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_nombre,
                "Capacidad": "",
                "CodigoProducto": extraer_codigo_producto(url_cel),
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount"
            })
driver.quit()

def scrape_page(page):
    driver = webdriver.Chrome(options=options)
    url = f"https://www.liverpool.com.mx/tienda/celulares/N-S1sLjNksKoG%2BC2c1SDPsHEVNIqwseF0ax1%2Fc4tPD6RI%3D/page-{page}"
    driver.get(url)
    time.sleep(9)
    for _ in range(2):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    products = soup.find_all("li", class_="m-product__card")
    print(f"Productos encontrados: {len(products)}")
    page_data = []
    for product in products:
        marca = product.find("h3", class_="a-card-brand")
        nombre = product.find("h3", class_="card-title a-card-description")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue
        precio_normal = product.find("p", class_="a-card-price")
        precio_desc = product.find("p", class_="a-card-discount")
        a_tag = product.find("a", href=True)
        url_cel = ""
        if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
            url_cel = "https://www.liverpool.com.mx" + a_tag["href"]
        color_div = product.find("div", class_="a-plp-color")
        colores = []
        if color_div:
            for span in color_div.find_all("span", class_="atom-color"):
                color_nombre = span.get("title", "")
                color_hex = span.get("data-color", "")
                colores.append((color_nombre, color_hex))
        else:
            colores = [("", "")]
        precio_normal_text = precio_normal.text.strip() if precio_normal else ""
        precio_desc_text = precio_desc.text.strip() if precio_desc else ""
        if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount"
                })
        elif ('-' in precio_normal_text or '-' in precio_desc_text) and url_cel not in visited_urls:
            variantes = extraer_detalle_producto(url_cel, options)
            visited_urls.add(url_cel)
            combinaciones_unicas = set()
            for color_det, codigo_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
                clave = (color_det, tamano_det, extraer_sku(url_cel))
                if clave in combinaciones_unicas:
                    continue
                combinaciones_unicas.add(clave)
                if not precio_normal_det and precio_desc_det:
                    precio_normal_final = formatear_precio(precio_desc_det)
                    precio_desc_final = "No discount"
                else:
                    precio_normal_final = formatear_precio(precio_normal_det)
                    precio_desc_final = formatear_precio(precio_desc_det) if precio_desc_det else "No discount"
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_det,
                    "Capacidad": tamano_det,
                    "CodigoProducto": codigo_det,
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": precio_normal_final,
                    "Precio_descuento": precio_desc_final
                })
        else:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount"
                })
    driver.quit()
    return page_data

pages = list(range(2, 5))
with ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(scrape_page, pages)
    for page_data in results:
        final_data.extend(page_data)

if final_data:
    df = pd.DataFrame(final_data)
    diccionario = pd.read_csv("diccionario liverpool.csv", encoding="utf-8")
    if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
        diccionario.rename(columns={"sku": "SKU"}, inplace=True)
    if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
        diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
        df["SKU"] = df["SKU"].astype(str).str.strip()
        diccionario = diccionario[["SKU", "Capacidad"]]
        df = df.merge(diccionario, on="SKU", how="left")
        df["Capacidad"] = df["Capacidad_y"].combine_first(df["Capacidad_x"])
        df.drop(columns=[col for col in df.columns if col.startswith("Capacidad_")], inplace=True)
    else:
        print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'. No se agregará capacidad por SKU.")
    
    # Calcula el porcentaje de descuento
    df["Porcentaje_descuento"] = df.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )
    
    columnas_final = [
            "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
            "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento"
        ]
    df = df[columnas_final]

    # Elimina duplicados
    df = df.drop_duplicates()

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_colores_{fecha_hoy} V2.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"Datos exportados a {nombre_archivo}")
else:
    print("No se encontraron productos. Ajusta el selector de productos si es necesario.")

Productos encontrados: 56
Productos encontrados: 0
Productos encontrados: 55
Productos encontrados: 56
Datos exportados a liverpool_colores_2026-05-14 V2.xlsx


# Extraer Tamaño/Capacidad desde URLs Liverpool CSV
Lee las URLs del archivo `urls liverpool.csv` y extrae el tamaño (capacidad) de cada celular visitando la página de detalle.

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ===================== CONFIGURACIÓN =====================

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

# ===================== LEER URLs DEL CSV =====================

csv_path = r"c:\Users\c50012218\Downloads\urls liverpool.csv"
df_urls = pd.read_csv(csv_path, header=None, names=["URL"])
urls = df_urls["URL"].dropna().str.strip().tolist()
print(f"Total de URLs a procesar: {len(urls)}")

# ===================== FUNCIÓN PARA EXTRAER TAMAÑO DE UNA URL =====================

def extraer_tamano_url(url):
    """Visita la URL del producto y extrae todos los tamaños disponibles."""
    resultados = []
    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h3.attributeAccordionText"))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # Extraer nombre y marca
        nombre_tag = soup.find("h1", class_="a-product__information--title")
        nombre = nombre_tag.get_text(strip=True) if nombre_tag else ""
        marca_tag = soup.find("a", class_="a-product__information--brand")
        marca = marca_tag.get_text(strip=True) if marca_tag else ""
        
        # Extraer SKU de la URL
        sku_match = re.search(r'skuid=(\d+)', url)
        sku = sku_match.group(1) if sku_match else ""
        codigo_match = re.search(r'/(\d+)\?', url)
        codigo = codigo_match.group(1) if codigo_match else ""
        
        # Buscar el selector de tamaños
        size_list = soup.find("ul", id="size-list-container")
        
        if size_list:
            size_buttons = size_list.find_all("button", class_="a-btn--actionpdp")
            for size_btn in size_buttons:
                tamano_text = size_btn.get_text(strip=True)
                size_skuid = size_btn.get("data-skuid", "")
                is_selected = "a-btn--actionpdp-active" in size_btn.get("class", [])
                resultados.append({
                    "Marca": marca,
                    "Nombre": nombre,
                    "CodigoProducto": codigo,
                    "SKU": size_skuid if size_skuid else sku,
                    "Tamano": tamano_text,
                    "URL": url
                })
        else:
            # No hay selector de tamaños, buscar en los atributos del accordion
            tamano = ""
            for h3 in soup.find_all("h3", class_="attributeAccordionText"):
                if "Tamaño:" in h3.text:
                    tamano_span = h3.find("span", class_="attributeAccordionTextValue")
                    tamano = tamano_span.text.strip() if tamano_span else ""
                    break
            resultados.append({
                "Marca": marca,
                "Nombre": nombre,
                "CodigoProducto": codigo,
                "SKU": sku,
                "Tamano": tamano,
                "URL": url
            })
        
        driver.quit()
    except Exception as e:
        print(f"  [ERROR] {url[:80]}... -> {e}")
        if driver:
            try:
                driver.quit()
            except:
                pass
        # Agregar entrada con tamaño vacío para no perder la URL
        sku_match = re.search(r'skuid=(\d+)', url)
        sku = sku_match.group(1) if sku_match else ""
        resultados.append({
            "Marca": "",
            "Nombre": "",
            "CodigoProducto": "",
            "SKU": sku,
            "Tamano": "",
            "URL": url
        })
    return resultados

# ===================== PROCESAR EN PARALELO (BATCHES DE 2) =====================

BATCH_SIZE = 2
all_results = []
total = len(urls)

print(f"\n=== Procesando {total} URLs en paralelo (batches de {BATCH_SIZE}) ===\n")

for i in range(0, total, BATCH_SIZE):
    batch = urls[i:i+BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"Batch {batch_num}/{total_batches} ({i+1}-{min(i+BATCH_SIZE, total)} de {total})...")
    
    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = {executor.submit(extraer_tamano_url, url): url for url in batch}
        for future in as_completed(futures):
            url = futures[future]
            try:
                resultado = future.result()
                all_results.extend(resultado)
            except Exception as e:
                print(f"  [ERROR] Batch error para {url[:60]}...: {e}")
    
    print(f"  Acumulado: {len(all_results)} registros")

# ===================== EXPORTAR RESULTADOS =====================

if all_results:
    df_tamanos = pd.DataFrame(all_results)
    df_tamanos = df_tamanos.drop_duplicates()
    
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_tamanos_{fecha_hoy}.xlsx"
    df_tamanos.to_excel(nombre_archivo, index=False)
    
    print(f"\n{'='*60}")
    print(f"RESUMEN")
    print(f"{'='*60}")
    print(f"URLs procesadas: {total}")
    print(f"Registros totales (con variantes de tamaño): {len(df_tamanos)}")
    print(f"Con tamaño encontrado: {len(df_tamanos[df_tamanos['Tamano'] != ''])}")
    print(f"Sin tamaño: {len(df_tamanos[df_tamanos['Tamano'] == ''])}")
    print(f"Exportado a: {nombre_archivo}")
    print(f"\nPrimeros registros:")
    print(df_tamanos.head(10).to_string(index=False))
else:
    print("No se encontraron resultados.")

Total de URLs a procesar: 66

=== Procesando 66 URLs en paralelo (batches de 2) ===

Batch 1/33 (1-2 de 66)...
  Acumulado: 2 registros
Batch 2/33 (3-4 de 66)...
  Acumulado: 4 registros
Batch 3/33 (5-6 de 66)...
  Acumulado: 6 registros
Batch 4/33 (7-8 de 66)...
  Acumulado: 8 registros
Batch 5/33 (9-10 de 66)...
  Acumulado: 10 registros
Batch 6/33 (11-12 de 66)...
  Acumulado: 12 registros
Batch 7/33 (13-14 de 66)...
  Acumulado: 14 registros
Batch 8/33 (15-16 de 66)...
  Acumulado: 16 registros
Batch 9/33 (17-18 de 66)...
  Acumulado: 18 registros
Batch 10/33 (19-20 de 66)...
  Acumulado: 20 registros
Batch 11/33 (21-22 de 66)...
  [ERROR] https://www.liverpool.com.mx/tienda/pdp/samsung-galaxy-s10e-de-128gb-amoled-5.8-... -> Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6892f10b5+14cc5]
	chromedriver!GetHandleVerifier [0x7ff6892f1120+14d30]
	chromedriver!(No symbol) [0x7ff68905db6d]
	chromedriver!(No symbol) [0x7ff6890b7082]
	chromedriver!(No symbol) [0x7ff6890b738c]


# Scraping por Vendedores (sin Liverpool)
Extrae celulares de la categoría general cat5150024, iterando por cada filtro de vendedor (excepto Liverpool).
Agrega una columna "Vendedor" con el nombre del filtro seleccionado.

In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

# ===================== FUNCIONES AUXILIARES =====================

def formatear_precio(precio_raw):
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    else:
        return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    else:
        return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def extraer_detalle_producto(url, options):
    resultados = []
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h3.attributeAccordionText"))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")
        color_list = soup.find("ul", id="color-list-mobile")
        color_buttons = color_list.find_all("a", class_="atom-color") if color_list else [None]
        size_list = soup.find("ul", id="size-list-container")
        size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]

        combinaciones_unicas = set()
        if color_buttons == [None] and size_buttons == [None]:
            color = ""
            codigo = ""
            tamano = ""
            precio_normal = ""
            precio_desc = ""
            color_tag = soup.find("h3", class_="attributeAccordionText")
            if color_tag and "Color:" in color_tag.text:
                color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                color = color_span.text.strip() if color_span else ""
            codigo_tag = soup.find("p", class_="product-header-subcontainer-information--code")
            if codigo_tag:
                codigo_span = codigo_tag.find("span")
                codigo = codigo_span.text.strip() if codigo_span else ""
            tamano_tag = None
            for h3 in soup.find_all("h3", class_="attributeAccordionText"):
                if "Tamaño:" in h3.text:
                    tamano_tag = h3
                    break
            if tamano_tag:
                tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                tamano = tamano_span.text.strip() if tamano_span else ""
            precio_normal_tag = soup.find("p", class_="a-product__paragraphRegularPrice")
            precio_desc_tag = soup.find("p", class_="a-product__paragraphDiscountPrice")
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            clave = (color, tamano, codigo)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((color, codigo, tamano, precio_normal, precio_desc))
        else:
            for color_btn in color_buttons:
                if color_btn:
                    color_skuid = color_btn.get("data-skuid")
                    try:
                        driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//a[@data-skuid='{color_skuid}']"))
                        time.sleep(2)
                    except Exception:
                        pass
                soup_color = BeautifulSoup(driver.page_source, "html.parser")
                size_list = soup_color.find("ul", id="size-list-container")
                size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]
                for size_btn in size_buttons:
                    if size_btn and '-disabled' not in size_btn.get('class', []):
                        size_skuid = size_btn.get("data-skuid")
                        try:
                            driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//button[@data-skuid='{size_skuid}']"))
                            time.sleep(2)
                        except Exception:
                            pass
                    soup_var = BeautifulSoup(driver.page_source, "html.parser")
                    if soup_var.find("p", class_="a-productInfo_selledByLink"):
                        continue
                    color = ""
                    codigo = ""
                    tamano = ""
                    precio_normal = ""
                    precio_desc = ""
                    color_tag = soup_var.find("h3", class_="attributeAccordionText")
                    if color_tag and "Color:" in color_tag.text:
                        color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                        color = color_span.text.strip() if color_span else ""
                    codigo_tag = soup_var.find("p", class_="product-header-subcontainer-information--code")
                    if codigo_tag:
                        codigo_span = codigo_tag.find("span")
                        codigo = codigo_span.text.strip() if codigo_span else ""
                    tamano_tag = None
                    for h3 in soup_var.find_all("h3", class_="attributeAccordionText"):
                        if "Tamaño:" in h3.text:
                            tamano_tag = h3
                            break
                    if tamano_tag:
                        tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                        tamano = tamano_span.text.strip() if tamano_span else ""
                    precio_normal_tag = soup_var.find("p", class_="a-product__paragraphRegularPrice")
                    precio_desc_tag = soup_var.find("p", class_="a-product__paragraphDiscountPrice")
                    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
                    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
                    clave = (color, tamano, codigo)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((color, codigo, tamano, precio_normal, precio_desc))
        driver.quit()
    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    return resultados

def es_smartwatch(nombre):
    nombre = (nombre or "").lower()
    palabras = ["smartwatch"]
    return any(p in nombre for p in palabras)

# ===================== CONFIGURACIÓN =====================

BASE_URL = "https://www.liverpool.com.mx/tienda/celulares/cat5150024"

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data_vendedores = []
visited_urls_vendedores = set()

# ===================== PASO 1: OBTENER FILTROS DE "Vendido por" =====================

print("=== Obteniendo filtros de 'Vendido por' ===")
driver = webdriver.Chrome(options=options)
driver.get(BASE_URL)
time.sleep(12)

# 1. Buscar y hacer clic en el botón "Vendido por" para expandir la sección
try:
    vendido_label = driver.find_element(By.CSS_SELECTOR, 'label[title="Vendido por"]')
    vendido_button = vendido_label.find_element(By.XPATH, "ancestor::button")
    driver.execute_script("arguments[0].click();", vendido_button)
    time.sleep(3)
    print("  Sección 'Vendido por' expandida.")
except Exception as e:
    print(f"  Nota: No se pudo hacer clic en 'Vendido por' (puede que ya esté expandida): {e}")

# 2. Hacer clic en "Ver más" si existe para mostrar todos los vendedores
try:
    ver_mas_buttons = driver.find_elements(By.CSS_SELECTOR, "button.a-plp-link-viewMore, a.a-plp-link-viewMore")
    for btn in ver_mas_buttons:
        try:
            parent = btn.find_element(By.XPATH, "..")
            parent_text = parent.text.lower()
            if "vendido" in parent_text or "seller" in parent_text:
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(2)
                print("  'Ver más' clickeado para vendedores.")
                break
        except:
            pass
    # Si no encontró con ese selector, intentar con texto
    if not ver_mas_buttons:
        all_buttons = driver.find_elements(By.TAG_NAME, "button")
        for btn in all_buttons:
            if "ver más" in btn.text.lower() or "ver mas" in btn.text.lower():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(2)
                print("  'Ver más' clickeado.")
                break
except Exception:
    pass

# 3. Parsear y extraer vendedores
time.sleep(2)
soup = BeautifulSoup(driver.page_source, "html.parser")

vendedores_info = []

# DEBUG: Guardar HTML de filtros para diagnóstico
print("\n  [DEBUG] Buscando checkboxes con id que contenga 'sellernames'...")

# Estrategia principal: buscar todos los inputs con id que contenga "sellernames"
all_seller_inputs = soup.find_all("input", id=lambda x: x and "sellernames" in x.lower())
print(f"  [DEBUG] Inputs con 'sellernames' encontrados: {len(all_seller_inputs)}")

for inp in all_seller_inputs:
    checkbox_id = inp.get("id", "")
    # Buscar el label hermano más cercano
    parent = inp.find_parent("div", class_="mdc-form-field")
    if not parent:
        parent = inp.find_parent("div")
    label = parent.find("label") if parent else None
    if label:
        label_text = label.get_text(strip=True)
        nombre_filtro = re.sub(r'\s*\(\d+\)\s*$', '', label_text).strip()
        if nombre_filtro and "liverpool" not in nombre_filtro.lower():
            vendedores_info.append({
                "nombre": nombre_filtro,
                "checkbox_id": checkbox_id
            })

# Si no encontró con sellernames, buscar por VendidoporcountViewMore
if not vendedores_info:
    print("  [DEBUG] No se encontraron por 'sellernames', buscando por 'VendidoporcountViewMore'...")
    vendedor_divs = soup.find_all("div", id=lambda x: x and "Vendidopor" in str(x))
    print(f"  [DEBUG] Divs con 'Vendidopor' encontrados: {len(vendedor_divs)}")
    for vdiv in vendedor_divs:
        checkbox = vdiv.find("input", class_="a-checkbox__input")
        label = vdiv.find("label")
        if checkbox and label:
            checkbox_id = checkbox.get("id", "")
            label_text = label.get_text(strip=True)
            nombre_filtro = re.sub(r'\s*\(\d+\)\s*$', '', label_text).strip()
            if nombre_filtro and "liverpool" not in nombre_filtro.lower():
                vendedores_info.append({
                    "nombre": nombre_filtro,
                    "checkbox_id": checkbox_id
                })

# Si aún no encontró, buscar por filter-brands
if not vendedores_info:
    print("  [DEBUG] Buscando en div.filter-brands...")
    filter_brands = soup.find("div", class_="filter-brands")
    if filter_brands:
        checkboxes = filter_brands.find_all("input", class_="a-checkbox__input")
        print(f"  [DEBUG] Checkboxes en filter-brands: {len(checkboxes)}")
        for cb in checkboxes:
            checkbox_id = cb.get("id", "")
            parent_div = cb.find_parent("div", class_="mdc-form-field")
            if not parent_div:
                parent_div = cb.find_parent("div")
            label = parent_div.find("label") if parent_div else None
            if label:
                label_text = label.get_text(strip=True)
                nombre_filtro = re.sub(r'\s*\(\d+\)\s*$', '', label_text).strip()
                if nombre_filtro and "liverpool" not in nombre_filtro.lower():
                    vendedores_info.append({
                        "nombre": nombre_filtro,
                        "checkbox_id": checkbox_id
                    })
    else:
        print("  [DEBUG] No se encontró div.filter-brands")

# Eliminar duplicados
vendedores_unicos = {}
for v in vendedores_info:
    if v["nombre"] not in vendedores_unicos:
        vendedores_unicos[v["nombre"]] = v
vendedores_info = list(vendedores_unicos.values())

# DEBUG: Imprimir sección del HTML relevante si no encontró nada
if not vendedores_info:
    print("\n  [DEBUG] No se encontraron vendedores. Buscando secciones de filtro...")
    filter_sections = soup.find_all("div", class_=lambda c: c and "plp-filter" in c)
    for fs in filter_sections[:5]:
        text = fs.get_text(strip=True)[:200]
        print(f"    Sección: {text}")
    # Buscar todos los labels con title
    labels_with_title = soup.find_all("label", attrs={"title": True})
    for lbl in labels_with_title:
        print(f"    Label con title: '{lbl.get('title')}' -> texto: '{lbl.get_text(strip=True)}'")

driver.quit()

print(f"\nVendedores encontrados (sin Liverpool): {len(vendedores_info)}")
for v in vendedores_info:
    print(f"  - {v['nombre']} (checkbox: {v['checkbox_id']})")

# ===================== PASO 2: SCRAPING POR VENDEDOR =====================

def scrape_productos_pagina(soup_page, vendedor_nombre):
    """Extrae productos de una página. vendedor_nombre es el nombre del filtro usado."""
    page_data = []
    products = soup_page.find_all("li", class_="m-product__card")
    if not products:
        products = soup_page.find_all("div", class_="m-plp-card-container")
    
    for product in products:
        marca = product.find("h3", class_="a-card-brand")
        nombre = product.find("h3", class_="card-title a-card-description")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue
        precio_normal = product.find("p", class_="a-card-price")
        precio_desc = product.find("p", class_="a-card-discount")
        a_tag = product.find("a", href=True)
        url_cel = ""
        if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
            url_cel = "https://www.liverpool.com.mx" + a_tag["href"]
        color_div = product.find("div", class_="a-plp-color")
        colores = []
        if color_div:
            for span in color_div.find_all("span", class_="atom-color"):
                color_nombre = span.get("title", "")
                colores.append((color_nombre, ""))
        else:
            colores = [("", "")]
        precio_normal_text = precio_normal.text.strip() if precio_normal else ""
        precio_desc_text = precio_desc.text.strip() if precio_desc else ""
        
        if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
            for color_nombre, _ in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount",
                    "Vendedor": vendedor_nombre
                })
        elif ('-' in precio_normal_text or '-' in precio_desc_text) and url_cel not in visited_urls_vendedores:
            variantes = extraer_detalle_producto(url_cel, options)
            visited_urls_vendedores.add(url_cel)
            combinaciones_unicas = set()
            for color_det, codigo_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
                clave = (color_det, tamano_det, extraer_sku(url_cel))
                if clave in combinaciones_unicas:
                    continue
                combinaciones_unicas.add(clave)
                if not precio_normal_det and precio_desc_det:
                    precio_normal_final = formatear_precio(precio_desc_det)
                    precio_desc_final = "No discount"
                else:
                    precio_normal_final = formatear_precio(precio_normal_det)
                    precio_desc_final = formatear_precio(precio_desc_det) if precio_desc_det else "No discount"
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_det,
                    "Capacidad": tamano_det,
                    "CodigoProducto": codigo_det,
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": precio_normal_final,
                    "Precio_descuento": precio_desc_final,
                    "Vendedor": vendedor_nombre
                })
        else:
            for color_nombre, _ in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount",
                    "Vendedor": vendedor_nombre
                })
    return page_data

# ===================== ITERAR POR CADA VENDEDOR (PARALELO DE 2) =====================

import threading

# Lock para acceso thread-safe a las listas compartidas
lock = threading.Lock()

def procesar_vendedor(vendedor):
    """Procesa un vendedor completo: abre Chrome, hace clic en filtro, scrapea todas las páginas."""
    vendedor_nombre = vendedor["nombre"]
    vendedor_checkbox_id = vendedor["checkbox_id"]
    resultados_vendedor = []
    visited_local = set()
    
    print(f"\n{'='*60}")
    print(f"[INICIO] Procesando filtro: {vendedor_nombre}")
    print(f"{'='*60}")
    
    page = 1
    max_pages = 20
    filter_url = None
    
    try:
        driver = webdriver.Chrome(options=options)
    except Exception as e:
        print(f"  [ERROR] No se pudo abrir Chrome para {vendedor_nombre}: {e}")
        return resultados_vendedor
    
    try:
        driver.get(BASE_URL)
        time.sleep(10)
        
        try:
            checkbox = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, vendedor_checkbox_id))
            )
            driver.execute_script("arguments[0].click();", checkbox)
            time.sleep(8)
            filter_url = driver.current_url
        except Exception as e:
            print(f"  [ERROR] No se pudo hacer clic en checkbox de {vendedor_nombre} ({vendedor_checkbox_id}): {e}")
            driver.quit()
            return resultados_vendedor
        
        while page <= max_pages:
            if page > 1:
                if "/page-" not in filter_url:
                    page_url = filter_url + f"/page-{page}"
                else:
                    page_url = re.sub(r'/page-\d+', f'/page-{page}', filter_url)
                driver.get(page_url)
                time.sleep(10)
            
            for _ in range(3):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1.5)
            
            soup = BeautifulSoup(driver.page_source, "html.parser")
            
            # Extraer productos de la página
            page_data = []
            products = soup.find_all("li", class_="m-product__card")
            if not products:
                products = soup.find_all("div", class_="m-plp-card-container")
            
            for product in products:
                marca = product.find("h3", class_="a-card-brand")
                nombre = product.find("h3", class_="card-title a-card-description")
                nombre_texto = nombre.text.strip() if nombre else ""
                if es_smartwatch(nombre_texto):
                    continue
                precio_normal = product.find("p", class_="a-card-price")
                precio_desc = product.find("p", class_="a-card-discount")
                a_tag = product.find("a", href=True)
                url_cel = ""
                if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
                    url_cel = "https://www.liverpool.com.mx" + a_tag["href"]
                color_div = product.find("div", class_="a-plp-color")
                colores = []
                if color_div:
                    for span in color_div.find_all("span", class_="atom-color"):
                        color_nombre = span.get("title", "")
                        colores.append((color_nombre, ""))
                else:
                    colores = [("", "")]
                precio_normal_text = precio_normal.text.strip() if precio_normal else ""
                precio_desc_text = precio_desc.text.strip() if precio_desc else ""
                
                if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
                    for color_nombre, _ in colores:
                        page_data.append({
                            "Marca": marca.text.strip() if marca else "",
                            "Nombre": nombre.text.strip() if nombre else "",
                            "Color": color_nombre,
                            "Capacidad": "",
                            "CodigoProducto": extraer_codigo_producto(url_cel),
                            "SKU": extraer_sku(url_cel),
                            "URL": url_cel,
                            "FechaConsulta": fecha_consulta,
                            "Precio_normal": formatear_precio(precio_desc_text),
                            "Precio_descuento": "No discount",
                            "Vendedor": vendedor_nombre
                        })
                elif ('-' in precio_normal_text or '-' in precio_desc_text) and url_cel not in visited_local:
                    with lock:
                        already_visited = url_cel in visited_urls_vendedores
                    if not already_visited:
                        variantes = extraer_detalle_producto(url_cel, options)
                        visited_local.add(url_cel)
                        with lock:
                            visited_urls_vendedores.add(url_cel)
                        combinaciones_unicas = set()
                        for color_det, codigo_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
                            clave = (color_det, tamano_det, extraer_sku(url_cel))
                            if clave in combinaciones_unicas:
                                continue
                            combinaciones_unicas.add(clave)
                            if not precio_normal_det and precio_desc_det:
                                precio_normal_final = formatear_precio(precio_desc_det)
                                precio_desc_final = "No discount"
                            else:
                                precio_normal_final = formatear_precio(precio_normal_det)
                                precio_desc_final = formatear_precio(precio_desc_det) if precio_desc_det else "No discount"
                            page_data.append({
                                "Marca": marca.text.strip() if marca else "",
                                "Nombre": nombre.text.strip() if nombre else "",
                                "Color": color_det,
                                "Capacidad": tamano_det,
                                "CodigoProducto": codigo_det,
                                "SKU": extraer_sku(url_cel),
                                "URL": url_cel,
                                "FechaConsulta": fecha_consulta,
                                "Precio_normal": precio_normal_final,
                                "Precio_descuento": precio_desc_final,
                                "Vendedor": vendedor_nombre
                            })
                else:
                    for color_nombre, _ in colores:
                        page_data.append({
                            "Marca": marca.text.strip() if marca else "",
                            "Nombre": nombre.text.strip() if nombre else "",
                            "Color": color_nombre,
                            "Capacidad": "",
                            "CodigoProducto": extraer_codigo_producto(url_cel),
                            "SKU": extraer_sku(url_cel),
                            "URL": url_cel,
                            "FechaConsulta": fecha_consulta,
                            "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                            "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount",
                            "Vendedor": vendedor_nombre
                        })
            
            if not page_data:
                print(f"  [{vendedor_nombre}] Página {page}: Sin productos. Fin.")
                break
            
            print(f"  [{vendedor_nombre}] Página {page}: {len(page_data)} productos extraídos")
            resultados_vendedor.extend(page_data)
            
            next_page = soup.find("a", string=re.compile(r"Siguiente|Next", re.IGNORECASE))
            next_button = soup.find("button", string=re.compile(r"Siguiente|Next", re.IGNORECASE))
            pagination = soup.find("a", href=re.compile(f"page-{page+1}"))
            
            if not next_page and not next_button and not pagination:
                print(f"  [{vendedor_nombre}] No hay más páginas.")
                break
            
            page += 1
    
    except Exception as e:
        print(f"  [{vendedor_nombre}] Error inesperado: {e}")
    finally:
        try:
            driver.quit()
        except Exception:
            pass
    
    print(f"  [{vendedor_nombre}] Total: {len(resultados_vendedor)} productos")
    return resultados_vendedor

# Procesar vendedores en paralelo, 2 a la vez
BATCH_SIZE = 2
print(f"\n=== Procesando {len(vendedores_info)} vendedores en paralelo (batches de {BATCH_SIZE}) ===")

for i in range(0, len(vendedores_info), BATCH_SIZE):
    batch = vendedores_info[i:i+BATCH_SIZE]
    batch_nombres = [v["nombre"] for v in batch]
    print(f"\n>>> Batch {i//BATCH_SIZE + 1}: {batch_nombres}")
    
    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        resultados = list(executor.map(procesar_vendedor, batch))
    
    for resultado in resultados:
        final_data_vendedores.extend(resultado)
    
    print(f">>> Batch {i//BATCH_SIZE + 1} completado. Acumulado: {len(final_data_vendedores)} productos")

# ===================== EXPORTAR RESULTADOS =====================

if final_data_vendedores:
    df_vend = pd.DataFrame(final_data_vendedores)
    
    # Intentar cargar el diccionario para capacidades
    try:
        diccionario = pd.read_csv("diccionario liverpool 2.csv", encoding="utf-8")
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
            df_vend["SKU"] = df_vend["SKU"].astype(str).str.strip()
            diccionario = diccionario[["SKU", "Capacidad"]]
            df_vend = df_vend.merge(diccionario, on="SKU", how="left")
            df_vend["Capacidad"] = df_vend["Capacidad_y"].combine_first(df_vend["Capacidad_x"])
            df_vend.drop(columns=[col for col in df_vend.columns if col.startswith("Capacidad_")], inplace=True)
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
    except FileNotFoundError:
        print("Advertencia: No se encontró 'diccionario liverpool 2.csv'. Se continúa sin enriquecer capacidad.")
    
    # Calcular porcentaje de descuento
    df_vend["Porcentaje_descuento"] = df_vend.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )
    
    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento", "Vendedor"
    ]
    df_vend = df_vend[columnas_final]
    
    # Eliminar duplicados
    df_vend = df_vend.drop_duplicates()
    
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_vendedores_{fecha_hoy}.xlsx"
    df_vend.to_excel(nombre_archivo, index=False)
    
    print(f"\n{'='*60}")
    print(f"RESUMEN FINAL")
    print(f"{'='*60}")
    print(f"Total de productos: {len(df_vend)}")
    print(f"Vendedores (filtros) procesados: {df_vend['Vendedor'].nunique()}")
    for vend in df_vend['Vendedor'].unique():
        count = len(df_vend[df_vend['Vendedor'] == vend])
        print(f"  - {vend}: {count} productos")
    print(f"Datos exportados a {nombre_archivo}")
else:
    print("No se encontraron productos de vendedores externos.")

=== Obteniendo filtros de 'Vendido por' ===
  Sección 'Vendido por' expandida.

  [DEBUG] Buscando checkboxes con id que contenga 'sellernames'...
  [DEBUG] Inputs con 'sellernames' encontrados: 36

Vendedores encontrados (sin Liverpool): 35
  - TELMOV (checkbox: variants.sellernames-TELMOV)
  - GC Móvil (checkbox: variants.sellernames-GC Móvil)
  - TX COMMERCE (checkbox: variants.sellernames-TX COMMERCE)
  - RANOR INFINITY SA DE CV (checkbox: variants.sellernames-RANOR INFINITY SA DE CV)
  - commerweb (checkbox: variants.sellernames-commerweb)
  - Telcelcondesa (checkbox: variants.sellernames-Telcelcondesa)
  - CELUCENTER (checkbox: variants.sellernames-CELUCENTER)
  - Galeria Snack (checkbox: variants.sellernames-Galeria Snack)
  - Miami Wireless (checkbox: variants.sellernames-Miami Wireless)
  - Servicios y Consultoria Honupe (checkbox: variants.sellernames-Servicios y Consultoria Honupe)
  - MICEL COMUNICACIONES (checkbox: variants.sellernames-MICEL COMUNICACIONES)
  - LAFT (check

# Scraping Reacondicionados Liverpool
Scrapea la sección de reacondicionados de Liverpool, extrayendo además quién vende cada producto.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

def formatear_precio(precio_raw):
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    else:
        return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    else:
        return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    if not match:
        match = re.search(r'/(\d+)$', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def extraer_vendedor_de_soup(soup):
    """Extrae el vendedor desde el HTML de la página de detalle del producto."""
    vendedor = ""
    # Selector principal: <p class="a-productInfo_selledBy">Vendido por <a class="a-productInfo__selledByLink">...</a></p>
    vendedor_tag = soup.find("p", class_="a-productInfo_selledBy")
    if vendedor_tag:
        vendedor_link = vendedor_tag.find("a", class_="a-productInfo__selledByLink")
        if vendedor_link:
            vendedor = vendedor_link.text.strip()
        else:
            # Extraer texto después de "Vendido por"
            texto = vendedor_tag.get_text(strip=True)
            vendedor = texto.replace("Vendido por", "").replace("vendido por", "").strip()
    # Fallback: buscar con selectores alternativos
    if not vendedor:
        vendedor_tag = soup.find("a", class_="a-productInfo__selledByLink")
        vendedor = vendedor_tag.text.strip() if vendedor_tag else ""
    if not vendedor:
        vendedor_tag = soup.find("p", class_="a-productInfo_selledByLink")
        if vendedor_tag:
            link = vendedor_tag.find("a")
            vendedor = link.text.strip() if link else vendedor_tag.get_text(strip=True).replace("Vendido por", "").strip()
    if not vendedor:
        for tag in soup.find_all(["p", "span", "a"]):
            texto = tag.get_text(strip=True)
            if "Vendido por" in texto or "vendido por" in texto:
                vendedor = texto.replace("Vendido por", "").replace("vendido por", "").strip().lstrip(":").strip()
                break
    if not vendedor:
        vendedor = "Liverpool"
    return vendedor

def extraer_capacidad_nombre(nombre):
    """Extrae la capacidad de almacenamiento del nombre del producto."""
    if not nombre:
        return ""
    # Patrones comunes: "128 GB", "256GB", "1 TB", "128gb", "512 gb"
    match = re.search(r'(\d+)\s*(GB|TB|gb|tb|Gb|Tb)', nombre)
    if match:
        valor = match.group(1)
        unidad = match.group(2).upper()
        return f"{valor} {unidad}"
    return ""

def extraer_capacidad_de_soup(soup):
    """Extrae la capacidad desde la página de detalle del producto.
    Selector principal: <h3 class="attributeAccordionText">Tamaño: <span class="attributeAccordionTextValue">128 GB</span></h3>
    """
    capacidad = ""
    # Método principal: buscar en h3.attributeAccordionText > span.attributeAccordionTextValue
    for h3 in soup.find_all("h3", class_="attributeAccordionText"):
        texto_h3 = h3.get_text(strip=True).lower()
        if any(k in texto_h3 for k in ["tamaño:", "capacidad:", "almacenamiento:", "storage:"]):
            span = h3.find("span", class_="attributeAccordionTextValue")
            if span:
                capacidad = span.text.strip()
                break
    # Fallback: buscar en la tabla de especificaciones
    if not capacidad:
        for row in soup.find_all("tr"):
            tds = row.find_all("td")
            if len(tds) >= 2:
                header_text = tds[0].get_text(strip=True).lower()
                if any(k in header_text for k in ["capacidad", "almacenamiento", "memoria interna", "storage", "tamaño"]):
                    val = tds[1].get_text(strip=True)
                    if re.search(r'\d+\s*(GB|TB)', val, re.IGNORECASE):
                        capacidad = val
                        break
    return capacidad

def extraer_vendedor_y_capacidad_pagina(url, options):
    """Abre la página de detalle de un producto y extrae el vendedor y la capacidad."""
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        # Esperar a que cargue el acordeón de atributos donde está "Tamaño: 128 GB"
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h3.attributeAccordionText"))
            )
        except Exception:
            # Si no aparece el acordeón, esperar al menos al título
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1, h2, h3"))
            )
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        vendedor = extraer_vendedor_de_soup(soup)
        capacidad = extraer_capacidad_de_soup(soup)
        driver.quit()
        return vendedor, capacidad
    except Exception as e:
        print(f"  Error extrayendo vendedor/capacidad de {url}: {e}")
        return "Liverpool", ""

def extraer_detalle_reacondicionado(url, options):
    """Extrae detalles de un producto reacondicionado, incluyendo vendedor."""
    resultados = []
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h3.attributeAccordionText"))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Extraer vendedor con la función centralizada
        vendedor = extraer_vendedor_de_soup(soup)

        color_list = soup.find("ul", id="color-list-mobile")
        color_buttons = color_list.find_all("a", class_="atom-color") if color_list else [None]
        size_list = soup.find("ul", id="size-list-container")
        size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]

        combinaciones_unicas = set()
        if color_buttons == [None] and size_buttons == [None]:
            color = ""
            codigo = ""
            tamano = ""
            precio_normal = ""
            precio_desc = ""
            color_tag = soup.find("h3", class_="attributeAccordionText")
            if color_tag and "Color:" in color_tag.text:
                color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                color = color_span.text.strip() if color_span else ""
            codigo_tag = soup.find("p", class_="product-header-subcontainer-information--code")
            if codigo_tag:
                codigo_span = codigo_tag.find("span")
                codigo = codigo_span.text.strip() if codigo_span else ""
            tamano_tag = None
            for h3 in soup.find_all("h3", class_="attributeAccordionText"):
                if "Tamaño:" in h3.text:
                    tamano_tag = h3
                    break
            if tamano_tag:
                tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                tamano = tamano_span.text.strip() if tamano_span else ""
            # Si no se encontró tamaño en el acordeón, usar función de extracción de capacidad
            if not tamano:
                tamano = extraer_capacidad_de_soup(soup)
            # Último fallback: extraer del título/nombre del producto
            if not tamano:
                titulo_tag = soup.find("h1") or soup.find("h2")
                if titulo_tag:
                    tamano = extraer_capacidad_nombre(titulo_tag.get_text(strip=True))
            precio_normal_tag = soup.find("p", class_="a-product__paragraphRegularPrice")
            precio_desc_tag = soup.find("p", class_="a-product__paragraphDiscountPrice")
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            clave = (color, tamano, codigo)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((color, codigo, tamano, precio_normal, precio_desc, vendedor))
        else:
            for color_btn in color_buttons:
                if color_btn:
                    color_skuid = color_btn.get("data-skuid")
                    try:
                        driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//a[@data-skuid='{color_skuid}']"))
                        time.sleep(2)
                    except Exception:
                        pass
                soup_color = BeautifulSoup(driver.page_source, "html.parser")
                size_list = soup_color.find("ul", id="size-list-container")
                size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]
                for size_btn in size_buttons:
                    if size_btn and '-disabled' not in size_btn.get('class', []):
                        size_skuid = size_btn.get("data-skuid")
                        try:
                            driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//button[@data-skuid='{size_skuid}']"))
                            time.sleep(2)
                        except Exception:
                            pass
                    soup_var = BeautifulSoup(driver.page_source, "html.parser")

                    # Re-extraer vendedor por variante con función centralizada
                    vendedor_var = extraer_vendedor_de_soup(soup_var)
                    if vendedor_var == "Liverpool" and vendedor != "Liverpool":
                        vendedor_var = vendedor  # Usar el vendedor general si no se encontró en la variante

                    color = ""
                    codigo = ""
                    tamano = ""
                    precio_normal = ""
                    precio_desc = ""
                    color_tag = soup_var.find("h3", class_="attributeAccordionText")
                    if color_tag and "Color:" in color_tag.text:
                        color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                        color = color_span.text.strip() if color_span else ""
                    codigo_tag = soup_var.find("p", class_="product-header-subcontainer-information--code")
                    if codigo_tag:
                        codigo_span = codigo_tag.find("span")
                        codigo = codigo_span.text.strip() if codigo_span else ""
                    tamano_tag = None
                    for h3 in soup_var.find_all("h3", class_="attributeAccordionText"):
                        if "Tamaño:" in h3.text:
                            tamano_tag = h3
                            break
                    if tamano_tag:
                        tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                        tamano = tamano_span.text.strip() if tamano_span else ""
                    # Si no se encontró tamaño en el acordeón, usar función de extracción
                    if not tamano:
                        tamano = extraer_capacidad_de_soup(soup_var)
                    if not tamano:
                        titulo_tag = soup_var.find("h1") or soup_var.find("h2")
                        if titulo_tag:
                            tamano = extraer_capacidad_nombre(titulo_tag.get_text(strip=True))
                    precio_normal_tag = soup_var.find("p", class_="a-product__paragraphRegularPrice")
                    precio_desc_tag = soup_var.find("p", class_="a-product__paragraphDiscountPrice")
                    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
                    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
                    clave = (color, tamano, codigo)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((color, codigo, tamano, precio_normal, precio_desc, vendedor_var))
        driver.quit()
    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    return resultados

def extraer_vendedor_card(product):
    """Intenta extraer el vendedor desde la card del listado (fallback)."""
    vendedor = ""
    # Selector correcto de la card
    vendedor_tag = product.find("p", class_="a-productInfo_selledBy")
    if vendedor_tag:
        link = vendedor_tag.find("a", class_="a-productInfo__selledByLink")
        if link:
            vendedor = link.text.strip()
        else:
            vendedor = vendedor_tag.get_text(strip=True).replace("Vendido por", "").strip()
    if not vendedor:
        vendedor_tag = product.find("a", class_="a-productInfo__selledByLink")
        vendedor = vendedor_tag.text.strip() if vendedor_tag else ""
    if not vendedor:
        vendedor_tag = product.find("p", class_="a-productInfo_selledByLink")
        if vendedor_tag:
            link = vendedor_tag.find("a")
            vendedor = link.text.strip() if link else vendedor_tag.get_text(strip=True)
    if not vendedor:
        for tag in product.find_all(["p", "span", "a"]):
            texto = tag.get_text(strip=True)
            if "Vendido por" in texto or "vendido por" in texto or "Sold by" in texto:
                vendedor = texto.replace("Vendido por", "").replace("vendido por", "").replace("Sold by", "").strip().lstrip(":").strip()
                break
    return vendedor

def es_smartwatch(nombre):
    nombre = (nombre or "").lower()
    palabras = ["smartwatch"]
    return any(p in nombre for p in palabras)

# ============================================================
# CONFIGURACIÓN
# ============================================================
BASE_URL = "https://www.liverpool.com.mx/tienda/reacondicionados/catst23690313"

options = Options()
# options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data_reac = []
visited_urls_reac = set()

# ============================================================
# FUNCIÓN PARA SCRAPEAR UNA PÁGINA (usada para todas las páginas)
# ============================================================
def scrape_page_reac(page):
    driver = webdriver.Chrome(options=options)
    # Página 1 usa la URL base, las demás usan /page-X
    if page == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}/page-{page}"
    driver.get(url)
    time.sleep(10)
    for _ in range(2):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    products = soup.find_all("li", class_="m-product__card")
    print(f"Página {page} - Productos encontrados: {len(products)}")
    page_data = []
    for product in products:
        marca = product.find("h3", class_="a-card-brand")
        nombre = product.find("h3", class_="card-title a-card-description")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue
        precio_normal = product.find("p", class_="a-card-price")
        precio_desc = product.find("p", class_="a-card-discount")
        a_tag = product.find("a", href=True)
        url_cel = ""
        if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
            url_cel = "https://www.liverpool.com.mx" + a_tag["href"]
        color_div = product.find("div", class_="a-plp-color")
        colores = []
        if color_div:
            for span in color_div.find_all("span", class_="atom-color"):
                color_nombre = span.get("title", "")
                color_hex = span.get("data-color", "")
                colores.append((color_nombre, color_hex))
        else:
            colores = [("", "")]
        precio_normal_text = precio_normal.text.strip() if precio_normal else ""
        precio_desc_text = precio_desc.text.strip() if precio_desc else ""
        
        # Intentar extraer vendedor de la card primero
        vendedor = extraer_vendedor_card(product)
        capacidad_detalle = ""
        
        # Entrar a la página del producto para obtener vendedor (si falta) y capacidad
        if url_cel and url_cel not in visited_urls_reac:
            if not vendedor:
                print(f"  Entrando a detalle para vendedor+capacidad: {nombre_texto[:50]}...")
            else:
                print(f"  Entrando a detalle para capacidad: {nombre_texto[:50]}...")
            vendedor_det, capacidad_detalle = extraer_vendedor_y_capacidad_pagina(url_cel, options)
            if not vendedor:
                vendedor = vendedor_det
        
        if not vendedor:
            vendedor = "Liverpool"
        
        # Extraer capacidad del nombre como fallback
        capacidad_nombre = extraer_capacidad_nombre(nombre_texto)
        # Prioridad: capacidad de la página de detalle > capacidad del nombre
        capacidad_final = capacidad_detalle if capacidad_detalle else capacidad_nombre

        if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": capacidad_final,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount",
                    "Vendedor": vendedor
                })
        elif ('-' in precio_normal_text or '-' in precio_desc_text) and url_cel not in visited_urls_reac:
            variantes = extraer_detalle_reacondicionado(url_cel, options)
            visited_urls_reac.add(url_cel)
            combinaciones_unicas = set()
            for color_det, codigo_det, tamano_det, precio_normal_det, precio_desc_det, vendedor_det in variantes:
                clave = (color_det, tamano_det, extraer_sku(url_cel))
                if clave in combinaciones_unicas:
                    continue
                combinaciones_unicas.add(clave)
                # Si el detalle no trajo capacidad, usar la extraída del nombre
                cap_variante = tamano_det if tamano_det else capacidad_final
                if not precio_normal_det and precio_desc_det:
                    precio_normal_final = formatear_precio(precio_desc_det)
                    precio_desc_final = "No discount"
                else:
                    precio_normal_final = formatear_precio(precio_normal_det)
                    precio_desc_final = formatear_precio(precio_desc_det) if precio_desc_det else "No discount"
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_det,
                    "Capacidad": cap_variante,
                    "CodigoProducto": codigo_det,
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": precio_normal_final,
                    "Precio_descuento": precio_desc_final,
                    "Vendedor": vendedor_det
                })
        else:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": capacidad_final,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount",
                    "Vendedor": vendedor
                })
    driver.quit()
    return page_data

# Procesar páginas en lotes de 2 para evitar MemoryError
lotes = [[1, 2], [3, 4]]
for lote in lotes:
    print(f"\n--- Procesando lote: páginas {lote} ---")
    with ThreadPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(scrape_page_reac, lote))
    for page_data in results:
        final_data_reac.extend(page_data)
    print(f"--- Lote páginas {lote} completado. Total acumulado: {len(final_data_reac)} ---")

# ============================================================
# PROCESAR Y EXPORTAR
# ============================================================
if final_data_reac:
    df_reac = pd.DataFrame(final_data_reac)

    # Intentar usar diccionario de capacidades
    try:
        diccionario = pd.read_csv("diccionario liverpool 2.csv", encoding="utf-8")
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
            df_reac["SKU"] = df_reac["SKU"].astype(str).str.strip()
            diccionario = diccionario[["SKU", "Capacidad"]]
            df_reac = df_reac.merge(diccionario, on="SKU", how="left")
            df_reac["Capacidad"] = df_reac["Capacidad_y"].combine_first(df_reac["Capacidad_x"])
            df_reac.drop(columns=[col for col in df_reac.columns if col.startswith("Capacidad_")], inplace=True)
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
    except FileNotFoundError:
        print("Advertencia: No se encontró 'diccionario liverpool 2.csv'. Se continúa sin enriquecer capacidad.")

    # Porcentaje de descuento
    df_reac["Porcentaje_descuento"] = df_reac.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )

    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento", "Vendedor"
    ]
    df_reac = df_reac[columnas_final]
    df_reac = df_reac.drop_duplicates()

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_reacondicionados_{fecha_hoy}.xlsx"
    df_reac.to_excel(nombre_archivo, index=False)
    print(f"\nResumen Reacondicionados:")
    print(f"Total de productos: {len(df_reac)}")
    print(f"Vendedores encontrados: {df_reac['Vendedor'].unique().tolist()}")
    print(f"Datos exportados a {nombre_archivo}")
else:
    print("No se encontraron productos reacondicionados.")


--- Procesando lote: páginas [1, 2] ---
Página 2 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: Redmi 9C IPS 6.5 pulgadas Desbloqueado Reacondicio...
Página 2 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: Redmi 9C IPS 6.5 pulgadas Desbloqueado Reacondicio...
Página 1 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: iPhone 14 Pro Max 6.7 pulgadas AMOLED Reacondicion...
Página 1 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: iPhone 14 Pro Max 6.7 pulgadas AMOLED Reacondicion...
  Entrando a detalle para vendedor+capacidad: Hyline Pro LCD IPS 6.6 pulgadas Reacondicionado...
  Entrando a detalle para vendedor+capacidad: Hyline Pro LCD IPS 6.6 pulgadas Reacondicionado...
  Entrando a detalle para vendedor+capacidad: Galaxy S24 Fe Dynamic AMOLED 2X 6.7 pulgadas Reaco...
  Entrando a detalle para vendedor+capacidad: Galaxy S24 Fe Dynamic AMOLED 2X 6.7 pulgadas Reaco...
  Entrando a deta

# Manual

In [1]:
# Versión modificada: No scrappear cuando hay "-" en el precio, escribir a mano en su lugar
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

def formatear_precio(precio_raw):
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    else:
        return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    else:
        return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def es_smartwatch(nombre):
    nombre = (nombre or "").lower()
    palabras = ["smartwatch"]
    return any(p in nombre for p in palabras)

options = Options()
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data = []
visited_urls = set()

driver = webdriver.Chrome(options=options)
driver.get("https://www.liverpool.com.mx/tienda/celulares/N-S1sLjNksKoG%2BC2c1SDPsHEVNIqwseF0ax1%2Fc4tPD6RI%3D")
time.sleep(10)
soup = BeautifulSoup(driver.page_source, "html.parser")
products = soup.find_all("li", class_="m-product__card")
print(f"Productos encontrados: {len(products)}")

for product in products:
    marca = product.find("h3", class_="a-card-brand")
    nombre = product.find("h3", class_="card-title a-card-description")
    nombre_texto = nombre.text.strip() if nombre else ""
    
    if es_smartwatch(nombre_texto):
        continue
    
    precio_normal = product.find("p", class_="a-card-price")
    precio_desc = product.find("p", class_="a-card-discount")
    a_tag = product.find("a", href=True)
    url_cel = ""
    
    if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
        url_cel = "https://www.liverpool.com.mx" + a_tag["href"]
    
    color_div = product.find("div", class_="a-plp-color")
    colores = []
    if color_div:
        for span in color_div.find_all("span", class_="atom-color"):
            color_nombre = span.get("title", "")
            color_hex = span.get("data-color", "")
            colores.append((color_nombre, color_hex))
    else:
        colores = [("", "")]
    
    precio_normal_text = precio_normal.text.strip() if precio_normal else ""
    precio_desc_text = precio_desc.text.strip() if precio_desc else ""
    
    # Si solo hay precio en descuento, guárdalo en precio_normal y "No discount" en descuento
    if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
        for color_nombre, color_hex in colores:
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_nombre,
                "Capacidad": "",
                "CodigoProducto": extraer_codigo_producto(url_cel),
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": formatear_precio(precio_desc_text),
                "Precio_descuento": "No discount"
            })
    # MODIFICACIÓN: Si hay "-" en el precio, escribir "Manual Entry" en lugar de scrappear
    elif '-' in precio_normal_text or '-' in precio_desc_text:
        for color_nombre, color_hex in colores:
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_nombre,
                "Capacidad": "Manual Entry",  # Indicar que requiere entrada manual
                "CodigoProducto": extraer_codigo_producto(url_cel),
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": "Manual Entry",  # Indicar que requiere entrada manual
                "Precio_descuento": "Manual Entry"  # Indicar que requiere entrada manual
            })
            print(f"Producto con precio variable encontrado: {nombre_texto} - Se requiere entrada manual")
    else:
        for color_nombre, color_hex in colores:
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_nombre,
                "Capacidad": "",
                "CodigoProducto": extraer_codigo_producto(url_cel),
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount"
            })

driver.quit()

def scrape_page_modified(page):
    """Función modificada para páginas adicionales sin scraping de detalles"""
    driver = webdriver.Chrome(options=options)
    url = f"https://www.liverpool.com.mx/tienda/celulares/N-S1sLjNksKoG%2BC2c1SDPsHEVNIqwseF0ax1%2Fc4tPD6RI%3D/page-{page}"
    driver.get(url)
    time.sleep(9)
    
    for _ in range(2):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    products = soup.find_all("li", class_="m-product__card")
    print(f"Página {page} - Productos encontrados: {len(products)}")
    
    page_data = []
    for product in products:
        marca = product.find("h3", class_="a-card-brand")
        nombre = product.find("h3", class_="card-title a-card-description")
        nombre_texto = nombre.text.strip() if nombre else ""
        
        if es_smartwatch(nombre_texto):
            continue
        
        precio_normal = product.find("p", class_="a-card-price")
        precio_desc = product.find("p", class_="a-card-discount")
        a_tag = product.find("a", href=True)
        url_cel = ""
        
        if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
            url_cel = "https://www.liverpool.com.mx" + a_tag["href"]
        
        color_div = product.find("div", class_="a-plp-color")
        colores = []
        if color_div:
            for span in color_div.find_all("span", class_="atom-color"):
                color_nombre = span.get("title", "")
                color_hex = span.get("data-color", "")
                colores.append((color_nombre, color_hex))
        else:
            colores = [("", "")]
        
        precio_normal_text = precio_normal.text.strip() if precio_normal else ""
        precio_desc_text = precio_desc.text.strip() if precio_desc else ""
        
        if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount"
                })
        # MODIFICACIÓN: Si hay "-" en el precio, escribir "Manual Entry" en lugar de scrappear
        elif '-' in precio_normal_text or '-' in precio_desc_text:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "Manual Entry",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": "Manual Entry",
                    "Precio_descuento": "Manual Entry"
                })
                print(f"Página {page} - Producto con precio variable: {nombre_texto} - Se requiere entrada manual")
        else:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount"
                })
    
    driver.quit()
    return page_data

# Procesar páginas adicionales con la función modificada
pages = list(range(2, 5))
with ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(scrape_page_modified, pages)
    for page_data in results:
        final_data.extend(page_data)

# Procesar los datos finales
if final_data:
    df = pd.DataFrame(final_data)
    
    # Intentar cargar el diccionario para capacidades
    try:
        diccionario = pd.read_csv("diccionario liverpool.csv", encoding="utf-8")
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
            df["SKU"] = df["SKU"].astype(str).str.strip()
            diccionario = diccionario[["SKU", "Capacidad"]]
            df = df.merge(diccionario, on="SKU", how="left")
            
            # Solo actualizar capacidad si no es "Manual Entry"
            df["Capacidad"] = df.apply(
                lambda row: row["Capacidad_y"] if pd.notna(row["Capacidad_y"]) and row["Capacidad_x"] != "Manual Entry"
                else row["Capacidad_x"], axis=1
            )
            df.drop(columns=[col for col in df.columns if col.startswith("Capacidad_")], inplace=True)
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
    except FileNotFoundError:
        print("Advertencia: No se encontró el archivo 'diccionario liverpool.csv'.")
    
    # Calcular porcentaje de descuento solo para precios válidos
    def calcular_porcentaje_seguro(row):
        if row["Precio_normal"] == "Manual Entry" or row["Precio_descuento"] == "Manual Entry":
            return "Manual Entry"
        else:
            return calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"])
    
    df["Porcentaje_descuento"] = df.apply(calcular_porcentaje_seguro, axis=1)
    
    # Organizar columnas finales
    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento"
    ]
    df = df[columnas_final]
    
    # Eliminar duplicados
    df = df.drop_duplicates()
    
    # Contar productos que requieren entrada manual
    productos_manuales = df[df["Precio_normal"] == "Manual Entry"].shape[0]
    print(f"\nResumen:")
    print(f"Total de productos procesados: {len(df)}")
    print(f"Productos que requieren entrada manual de precios: {productos_manuales}")
    
    # Guardar archivo
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_colores_{fecha_hoy}_sin_scraping.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"Datos exportados a {nombre_archivo}")
    
    # Mostrar algunos productos que requieren entrada manual
    if productos_manuales > 0:
        print(f"\nEjemplos de productos que requieren entrada manual:")
        manual_products = df[df["Precio_normal"] == "Manual Entry"][["Marca", "Nombre", "URL"]].head(5)
        for idx, row in manual_products.iterrows():
            print(f"- {row['Marca']} {row['Nombre']}")
else:
    print("No se encontraron productos. Ajusta el selector de productos si es necesario.")

Productos encontrados: 56
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy A16 Super AMOLED 6.7 pulgadas - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy A16 Super AMOLED 6.7 pulgadas - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy A16 Super AMOLED 6.7 pulgadas - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy S25 Ultra Dynamic AMOLED 2X 6.9 pulgadas - Se requiere 